# AI Image Generator Backend (WebUI Forge + Flux.1 + Google Drive)
Run this notebook to start your free cloud GPU backend. It will mount your Google Drive, download the massive 17GB Flux.1 model to it (only once!), and start WebUI Forge. When it's ready, paste the public Gradio URL into your custom Web App.

In [ ]:
import os
from google.colab import drive

# 1. Mount Google Drive to save the massive Flux model permanently
print('Mounting Google Drive...')
drive.mount('/content/drive')

drive_models_dir = '/content/drive/MyDrive/AI_Models/Stable-diffusion'
os.makedirs(drive_models_dir, exist_ok=True)

# 2. Download Flux.1 [dev] FP8 model to Google Drive if it doesn't exist
flux_model_path = os.path.join(drive_models_dir, 'flux1-dev-fp8.safetensors')
if not os.path.exists(flux_model_path):
    print('Downloading Flux.1 [dev] FP8 to your Google Drive (This will take a while, but only happens once!)...')
    !wget -q --show-progress -O {flux_model_path} https://huggingface.co/lllyasviel/flux1_dev/resolve/main/flux1-dev-fp8.safetensors
else:
    print('Flux.1 model found in your Google Drive! Skipping download.')

# 3. Clone WebUI Forge (Optimized for running Flux on 16GB GPUs like Colab T4)
if not os.path.exists('/content/stable-diffusion-webui-forge'):
    !git clone https://github.com/lllyasviel/stable-diffusion-webui-forge.git

%cd /content/stable-diffusion-webui-forge

# 4. Install Python 3.10 and set up venv (Fixes Colab Python 3.13 incompatibilities)
print('Setting up Python 3.10 environment...')
!sudo apt-get update -y > /dev/null 2>&1
!sudo apt-get install software-properties-common -y > /dev/null 2>&1
!sudo add-apt-repository ppa:deadsnakes/ppa -y > /dev/null 2>&1
!sudo apt-get install python3.10 python3.10-venv python3.10-dev -y > /dev/null 2>&1
if not os.path.exists('venv'):
    !python3.10 -m venv venv

# 5. Launch Forge API, pointing it to your Google Drive for models
print('Starting WebUI Forge...')
!COMMANDLINE_ARGS="--api --share --cors-allow-origins=* --enable-insecure-extension-access --gradio-queue --ckpt-dir {drive_models_dir}" MPLBACKEND="Agg" ./venv/bin/python launch.py